# Project 2 - train.py

Mark Khusid

In [1]:
import pandas as pd
import numpy as np

import pickle

from scipy.integrate import simpson

from sklearn.impute import KNNImputer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score

## Load the datasets

In [2]:
insulin_data_1 = pd.read_csv('InsulinData.csv', low_memory=False)
insulin_data_2 = pd.read_csv('Insulin_patient2.csv', low_memory=False)
cgm_data_1 = pd.read_csv('CGMData.csv', low_memory=False)
cgm_data_2 = pd.read_csv('CGM_patient2.csv', low_memory=False)

## View Dataframes

In [90]:
#insulin_data_1

In [4]:
#cgm_data_1

In [5]:
#insulin_data_2

In [6]:
#cgm_data_2

## EDA on dataframes

In [92]:
#insulin_data_1.info()

In [8]:
#cgm_data_1.info()

In [9]:
#insulin_data_2.info()

In [10]:
#cgm_data_2.info()

## Convert 'Date' and 'Time' columns to datetime format in both datasets for easier manipulation

In [11]:
def preprocess_date(date_string):
    date_string = date_string.strip()
    if len(date_string.split()) == 1:
        return date_string
    elif len(date_string.split()) == 2:
        return date_string.split()[0]

In [12]:
insulin_data_1['Date'] = insulin_data_1['Date'].apply(preprocess_date)
insulin_data_2['Date'] = insulin_data_2['Date'].apply(preprocess_date)

In [13]:
insulin_data_1['datetime'] = pd.to_datetime(insulin_data_1['Date'] + ' ' + insulin_data_1['Time'])
insulin_data_2['datetime'] = pd.to_datetime(insulin_data_2['Date'] + ' ' + insulin_data_2['Time'])

cgm_data_1['datetime'] = pd.to_datetime(cgm_data_1['Date'] + ' ' + cgm_data_1['Time'])
cgm_data_2['datetime'] = pd.to_datetime(cgm_data_2['Date'] + ' ' + cgm_data_2['Time'])

In [14]:
#insulin_data_1[['Date', 'Time', 'datetime']]

In [15]:
#insulin_data_2[['Date', 'Time', 'datetime']]

In [16]:
#cgm_data_1[['Date', 'Time', 'datetime']]

In [17]:
#cgm_data_2[['Date', 'Time', 'datetime']]

## Search for non-zero BMZ Carb Input Data

In [18]:
# This datetime series contains the start of all non NaN and  positive carb input
# data from the column BWZ Carb Input (grams)
meal_times_1 = \
    insulin_data_1[~insulin_data_1['BWZ Carb Input (grams)'].isna() & \
    (insulin_data_1['BWZ Carb Input (grams)'] > 0)]['datetime']
meal_times_1

48      2018-02-12 09:15:45
129     2018-02-12 02:30:55
188     2018-02-11 20:33:18
207     2018-02-11 18:14:37
222     2018-02-11 16:27:04
                ...        
41265   2017-07-26 11:24:52
41274   2017-07-26 09:27:16
41347   2017-07-25 18:31:40
41393   2017-07-25 10:39:46
41401   2017-07-25 10:21:19
Name: datetime, Length: 747, dtype: datetime64[ns]

In [19]:
# This datetime series contains the start of all non NaN and  positive carb input
# data from the column BWZ Carb Input (grams)
meal_times_2 = \
    insulin_data_2[~insulin_data_2['BWZ Carb Input (grams)'].isna() & \
    (insulin_data_2['BWZ Carb Input (grams)'] > 0)]['datetime']
#meal_times_2

## Extract Meal Data

### Obtain CGM Data based on Length of After-Meal Window

In [20]:
# Initialize meal data list
meal_data_1 = []

# Loop through each meal start time
for meal_time in meal_times_1:
    # A meal windows is 2 hours long
    meal_window = insulin_data_1[(insulin_data_1['datetime'] > meal_time) & 
                               (insulin_data_1['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
    #print(meal_window)

    # If that meal window is empty, get corresponding CGM_
    # data that starts at tm-30 and ends at tm+2hrs
    if meal_window.empty:
        # Step 4 condition (a): No meal between tm and tm+2hrs, use this stretch as meal data
        meal_cgm_data_1 = cgm_data_1[(cgm_data_1['datetime'] >= (meal_time - pd.Timedelta(minutes=30))) & 
                                 (cgm_data_1['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
    else:
        # Step 4 check for condition (b) or (c)
        next_meal_time = meal_window.iloc[0]['datetime']
        if next_meal_time < (meal_time + pd.Timedelta(hours=2)):
            # condition (b): There is a meal at tp between tm and tm+2hrs, use tp data
            meal_time = next_meal_time
            meal_cgm_data_1 = cgm_data_1[(cgm_data_1['datetime'] >= (meal_time - pd.Timedelta(minutes=30))) & 
                                     (cgm_data_1['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
        # Condition (c): There is a meal exactly at tm+2hrs, use tm+1hr 30min to tm+4hrs
        elif next_meal_time == (meal_time + pd.Timedelta(hours=2)):
            meal_cgm_data_1 = cgm_data_1[(cgm_data_1['datetime'] >= (meal_time + pd.Timedelta(hours=1, minutes=30))) & 
                                     (cgm_data_1['datetime'] <= (meal_time + pd.Timedelta(hours=4)))]

    # Check if the extracted data has the correct number of points (30 for meal data)
    if len(meal_cgm_data_1) == 30:
        #print(f"Meal Time Data {meal_time} has {len(meal_cgm_data)} number of points.")
        meal_data_1.append(meal_cgm_data_1['Sensor Glucose (mg/dL)'].values)

In [21]:
# Initialize meal data list
meal_data_2 = []

# Loop through each meal start time
for meal_time in meal_times_2:
    # A meal windows is 2 hours long
    meal_window = insulin_data_2[(insulin_data_2['datetime'] > meal_time) & 
                               (insulin_data_2['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
    #print(meal_window)

    # If that meal window is empty, get corresponding CGM_
    # data that starts at tm-30 and ends at tm+2hrs
    if meal_window.empty:
        # Step 4 condition (a): No meal between tm and tm+2hrs, use this stretch as meal data
        meal_cgm_data_2 = cgm_data_2[(cgm_data_2['datetime'] >= (meal_time - pd.Timedelta(minutes=30))) & 
                                 (cgm_data_2['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
    else:
        # Step 4 check for condition (b) or (c)
        next_meal_time = meal_window.iloc[0]['datetime']
        if next_meal_time < (meal_time + pd.Timedelta(hours=2)):
            # condition (b): There is a meal at tp between tm and tm+2hrs, use tp data
            meal_time = next_meal_time
            meal_cgm_data_2 = cgm_data_2[(cgm_data_2['datetime'] >= (meal_time - pd.Timedelta(minutes=30))) & 
                                     (cgm_data_2['datetime'] <= (meal_time + pd.Timedelta(hours=2)))]
        # Condition (c): There is a meal exactly at tm+2hrs, use tm+1hr 30min to tm+4hrs
        elif next_meal_time == (meal_time + pd.Timedelta(hours=2)):
            meal_cgm_data_2 = cgm_data_2[(cgm_data_2['datetime'] >= (meal_time + pd.Timedelta(hours=1, minutes=30))) & 
                                     (cgm_data_2['datetime'] <= (meal_time + pd.Timedelta(hours=4)))]

    # Check if the extracted data has the correct number of points (30 for meal data)
    if len(meal_cgm_data_2) == 30:
        #print(f"Meal Time Data {meal_time} has {len(meal_cgm_data)} number of points.")
        meal_data_2.append(meal_cgm_data_2['Sensor Glucose (mg/dL)'].values)

### Get Length of Meal Data CGM Values

In [22]:
len(meal_data_1)

715

In [23]:
len(meal_data_2)

372

In [88]:
meal_data_1[0:5]

[array([ nan,  nan, 124., 126., 127., 128., 130., 137., 140.,  nan, 150.,
        140., 131., 130., 134., 148.,  nan, 159., 157., 140., 146., 148.,
        151., 150., 162., 174., 180., 178., 184., 185.]),
 array([121., 120., 115., 110., 106., 103., 103., 101.,  nan, 131., 109.,
         82.,  74.,  66.,  73.,  82.,  87.,  90.,  83.,  73.,  49.,  58.,
         69.,  79.,  nan,  93.,  92.,  86.,  84.,  80.]),
 array([287., 288., 289., 279., 260., 262., 259., 250., 245., 250., 247.,
        243., 227., 213., 205., 182., 175., 170., 166., 164., 165., 158.,
        153., 161., 177., 194., 200., 201., 189., 162.]),
 array([189., 162., 166., 173., 176., 169., 167., 166., 168., 180., 182.,
        178., 171., 160., 145., 138., 132., 120., 127., 129., 132., 123.,
        104., 100., 107., 106., 123., 128., 137., 147.]),
 array([107., 106., 123., 128., 137., 147., 145., 142.,  nan,  nan,  nan,
         nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,
         nan,  nan,  nan,  n

In [25]:
#meal_data_2[0:5]

### Create Meal Data Matrix

In [26]:
meal_data_matrix_1 = np.array(meal_data_1)
meal_data_matrix_1

array([[ nan,  nan, 124., ..., 178., 184., 185.],
       [121., 120., 115., ...,  86.,  84.,  80.],
       [287., 288., 289., ..., 201., 189., 162.],
       ...,
       [274., 278., 283., ..., 268., 281., 292.],
       [223., 222., 222., ..., 324., 326., 328.],
       [ 82.,  85.,  92., ..., 164., 163., 155.]])

In [93]:
meal_data_matrix_1.shape

(715, 30)

In [27]:
meal_data_matrix_2 = np.array(meal_data_2)
meal_data_matrix_2

array([[390., 391., 389., ..., 165., 139., 133.],
       [116., 113., 110., ..., 248., 233., 222.],
       [130., 129., 126., ..., 153., 147., 147.],
       ...,
       [174., 181., 191., ..., 126., 129., 128.],
       [174., 181., 191., ..., 126., 129., 128.],
       [102.,  94.,  92., ..., 169., 180., 186.]])

In [28]:
meal_data_matrix_2.shape

(372, 30)

## Extract No - Meal Data

In [29]:
# Initialize meal data list
no_meal_data_1 = []
    
# Loop through each meal start time to find valid post - absorbtive windows
for meal_time in meal_times_1:
    # Define the post-absorptive period: starts from tm+2 hours
    # and lasts 2 hours (i.e to four hours)
    post_absorptive_start = meal_time + pd.Timedelta(hours=2)
    post_absorptive_end = meal_time + pd.Timedelta(hours=4)
    
    # Check for meals in the post-absorptive period
    post_absorptive_meals = insulin_data_1[(insulin_data_1['datetime'] >= post_absorptive_start) &
                                         (insulin_data_1['datetime'] <= post_absorptive_end)]
    
    # Step 2 Condition (a): If any non-zero, non-NaN meal is found in the
    # post-absorptive period, skip this stretch
    if not post_absorptive_meals.empty:
        # Check if any of the meals have a non-zero carb input
        if (post_absorptive_meals['BWZ Carb Input (grams)'] > 0).any():
            # Skip this stretch, as there is a meal found in the post-absorptive period
            continue
    
    # Step 2 condition (b): If meal values are 0 or NaN, ignore and consider the stretch
    # Extract the corresponding 2-hour CGM data
    no_meal_cgm_data_1 = cgm_data_1[(cgm_data_1['datetime'] >= post_absorptive_start) & 
                                (cgm_data_1['datetime'] <= post_absorptive_end)]
    
    # Check if the extracted data has the correct number of points (24 for no meal data)
    if len(no_meal_cgm_data_1) == 24:
        no_meal_data_1.append(no_meal_cgm_data_1['Sensor Glucose (mg/dL)'].values)

In [30]:
# Initialize meal data list
no_meal_data_2 = []
    
# Loop through each meal start time to find valid post - absorbtive windows
for meal_time in meal_times_2:
    # Define the post-absorptive period: starts from tm+2 hours
    # and lasts 2 hours (i.e to four hours)
    post_absorptive_start = meal_time + pd.Timedelta(hours=2)
    post_absorptive_end = meal_time + pd.Timedelta(hours=4)
    
    # Check for meals in the post-absorptive period
    post_absorptive_meals = insulin_data_2[(insulin_data_2['datetime'] >= post_absorptive_start) &
                                         (insulin_data_2['datetime'] <= post_absorptive_end)]
    
    # Step 2 Condition (a): If any non-zero, non-NaN meal is found in the
    # post-absorptive period, skip this stretch
    if not post_absorptive_meals.empty:
        # Check if any of the meals have a non-zero carb input
        if (post_absorptive_meals['BWZ Carb Input (grams)'] > 0).any():
            # Skip this stretch, as there is a meal found in the post-absorptive period
            continue
    
    # Step 2 condition (b): If meal values are 0 or NaN, ignore and consider the stretch
    # Extract the corresponding 2-hour CGM data
    no_meal_cgm_data_2 = cgm_data_2[(cgm_data_2['datetime'] >= post_absorptive_start) & 
                                (cgm_data_2['datetime'] <= post_absorptive_end)]
    
    # Check if the extracted data has the correct number of points (24 for no meal data)
    if len(no_meal_cgm_data_2) == 24:
        no_meal_data_2.append(no_meal_cgm_data_2['Sensor Glucose (mg/dL)'].values)

In [31]:
len(no_meal_data_1)

510

In [32]:
len(no_meal_data_2)

325

In [33]:
#no_meal_data_1[0:5]

In [34]:
#no_meal_data_2[0:5]

### Create No Meal Data Matrix

In [35]:
no_meal_data_matrix_1 = np.array(no_meal_data_1)
#no_meal_data_matrix_1

In [36]:
no_meal_data_matrix_2 = np.array(no_meal_data_2)
#no_meal_data_matrix_2

In [37]:
#no_meal_data_matrix_1.shape

In [38]:
#no_meal_data_matrix_2.shape

## Handle Missing Data

We will use a strategy of deleting rows of data that have more than 10% of bad data.  If less than 10%, we will use KNN with a nearest neighbor hyperparameter (K=7).

### Define Threshold for Dumping Data

In [39]:
# 10%
dump_thresold = 0.1

### Create KNN Imputer

In [40]:
knn_imputer = KNNImputer(n_neighbors=7)

### Parse Meal Data and Handle Missing Values

In [41]:
cleaned_meal_data_1 = []
for row in meal_data_matrix_1:

    proportion_missing_data = np.isnan(row).mean()

    if proportion_missing_data > dump_thresold:

        # Ignore this row
        #print(f"Ignoring row with {proportion_missing_data*100:.2f}% missing data")
        continue

    else:
        # The data is good, use it
        cleaned_meal_data_1.append(row)

In [42]:
cleaned_meal_data_2 = []
for row in meal_data_matrix_2:

    proportion_missing_data = np.isnan(row).mean()

    if proportion_missing_data > dump_thresold:

        # Ignore this row
        #print(f"Ignoring row with {proportion_missing_data*100:.2f}% missing data")
        continue

    else:
        # The data is good, use it
        cleaned_meal_data_2.append(row)

In [43]:
cleaned_meal_data_1[0]

array([121., 120., 115., 110., 106., 103., 103., 101.,  nan, 131., 109.,
        82.,  74.,  66.,  73.,  82.,  87.,  90.,  83.,  73.,  49.,  58.,
        69.,  79.,  nan,  93.,  92.,  86.,  84.,  80.])

In [44]:
cleaned_meal_data_2[0]

array([390., 391., 389., 383., 375., 368., 357., 349., 336., 326., 315.,
       305., 295., 286., 279., 273., 268., 259., 252., 246., 238., 229.,
       218., 206., 195., 192., 181., 165., 139., 133.])

### Convert Cleaned Meal Data List into a Matrix

In [45]:
cleaned_meal_data_matrix_1 = \
    np.array(cleaned_meal_data_1)

In [46]:
cleaned_meal_data_matrix_2 = \
    np.array(cleaned_meal_data_2)

### Use KNN to Impute NaNs in Cleaned Meal Data

In [47]:
cleaned_imputed_meal_matrix_1 = knn_imputer.fit_transform(cleaned_meal_data_matrix_1)

In [48]:
cleaned_imputed_meal_matrix_2 = knn_imputer.fit_transform(cleaned_meal_data_matrix_2)

In [49]:
#cleaned_imputed_meal_matrix_1[0]

In [50]:
#cleaned_imputed_meal_matrix_2[5]

### Parse No Meal Data and Handle Missing Values

In [51]:
cleaned_no_meal_data_1 = []
for row in no_meal_data_matrix_1:

    proportion_missing_data = np.isnan(row).mean()

    if proportion_missing_data > dump_thresold:

        # Ignore this row
        #print(f"Ignoring row with {proportion_missing_data*100:.2f}% missing data")
        continue

    else:
        # The data is good, use it
        cleaned_no_meal_data_1.append(row)

In [52]:
cleaned_no_meal_data_2 = []
for row in no_meal_data_matrix_2:

    proportion_missing_data = np.isnan(row).mean()

    if proportion_missing_data > dump_thresold:

        # Ignore this row
        #print(f"Ignoring row with {proportion_missing_data*100:.2f}% missing data")
        continue

    else:
        # The data is good, use it
        cleaned_no_meal_data_2.append(row)

In [53]:
cleaned_no_meal_data_1[0]

array([121., 120., 115., 110., 106., 103., 103., 101.,  nan, 131., 109.,
        82.,  74.,  66.,  73.,  82.,  87.,  90.,  83.,  73.,  49.,  58.,
        69.,  79.])

In [54]:
cleaned_no_meal_data_2[0]

array([390., 391., 389., 383., 375., 368., 357., 349., 336., 326., 315.,
       305., 295., 286., 279., 273., 268., 259., 252., 246., 238., 229.,
       218., 206.])

In [55]:
len(cleaned_no_meal_data_1[0])

24

In [56]:
len(cleaned_no_meal_data_2[0])

24

### Convert Cleaned No Meal Data List into a Matrix

In [57]:
cleaned_no_meal_data_matrix_1 = \
    np.array(cleaned_no_meal_data_1)

In [58]:
cleaned_no_meal_data_matrix_2 = \
    np.array(cleaned_no_meal_data_2)

### Use KNN to Impute NaNs in Cleaned No Meal Data

In [59]:
cleaned_imputed_no_meal_matrix_1 = knn_imputer.fit_transform(cleaned_no_meal_data_matrix_1)

In [60]:
cleaned_imputed_no_meal_matrix_2 = knn_imputer.fit_transform(cleaned_no_meal_data_matrix_2)

In [61]:
#cleaned_imputed_no_meal_matrix_1[0]

In [62]:
#cleaned_imputed_no_meal_matrix_2[0]

## Combine Patient 1 and Patient 2 Data

In [63]:
cleaned_imputed_meal_matrix = \
    np.vstack((cleaned_imputed_meal_matrix_1, cleaned_imputed_meal_matrix_2))

In [64]:
cleaned_imputed_no_meal_matrix = \
    np.vstack((cleaned_imputed_no_meal_matrix_1, cleaned_imputed_no_meal_matrix_2))

## Feature Extraction

### Define Feature Extractor Helper Functions

In [65]:
def compute_first_derivative(glucose_data, delta_t=5):
    return np.diff(glucose_data) / delta_t

In [66]:
def compute_second_derivative(glucose_values, delta_t=5):
    second_derivative = np.zeros(len(glucose_values) - 2)
    for i in range(1, len(glucose_values) - 1):
        second_derivative[i - 1] = \
            (glucose_values[i + 1] - 2 * glucose_values[i] + glucose_values[i - 1]) / (delta_t ** 2)
    return second_derivative

In [67]:
#first_derivative(cleaned_imputed_meal_matrix[0])

In [68]:
#second_derivative(cleaned_imputed_meal_matrix[0])

In [69]:
def compute_AUC(glucose_values, delta_t=5):
    return simpson(glucose_values, dx=delta_t)

### Define Feature Extractor Function

In [70]:
def feature_extractor(data_matrix):

    features = []

    for row in data_matrix:
        mean_glucose = np.mean(row)
        std_glucose  = np.std(row)
        max_glucose  = np.max(row)
        min_glucose  = np.min(row)
        
        first_derivative = compute_first_derivative(row)
        mean_first_derivate = np.mean(first_derivative)

        if len(row) > 2:
            second_derivative = compute_second_derivative(row)
            mean_second_derivative = np.mean(second_derivative)
        else:
            mean_second_derivative = 0

        AUC = compute_AUC(row)

        time_to_peak = np.argmax(row)

        features_vector = [
            mean_glucose,
            std_glucose,
            max_glucose,
            min_glucose,
            mean_first_derivate,
            mean_second_derivative,
            AUC,
            time_to_peak
        ]

        features.append(features_vector)
    
    return np.array(features)

### Get Feature Matrix for Cleaned Imputed Meal Data

In [71]:
meal_features_matrix = feature_extractor(cleaned_imputed_meal_matrix)
#len(meal_features_matrix[0])

### Get Feature Matrix for Clean Imputed No Meal Data

In [72]:
no_meal_features_matrix = feature_extractor(cleaned_imputed_no_meal_matrix)
#no_meal_features_matrix

## Concatenate Feature Matrices and Add Class Labels

In [73]:
meal_labels = np.ones((meal_features_matrix.shape[0], 1))
len(meal_labels)

944

In [74]:
meal_features_matrix_with_labels = \
    np.hstack((meal_features_matrix, meal_labels))
#meal_features_matrix_with_labels[1]

In [75]:
no_meal_labels = np.zeros((no_meal_features_matrix.shape[0], 1))
len(no_meal_labels)

752

In [76]:
no_meal_features_matrix_with_labels = \
    np.hstack((no_meal_features_matrix, no_meal_labels))
#no_meal_features_matrix_with_labels[1]

In [77]:
concat_features = \
    np.vstack((meal_features_matrix_with_labels, no_meal_features_matrix_with_labels))
len(concat_features)

1696

## Train Support Vector Machine

### Use Standard Variable Names for Features and Class Label

In [78]:
X = concat_features[:, :-1]
y = concat_features[:, -1]

### Train / Test Split

In [79]:
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size = 0.2, random_state=40)

### Normalilze The Data

In [80]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

### Train SVM Model

In [81]:
svm_model = SVC(kernel='linear', C=0.1)
svm_model.fit(X_train_scaled, y_train)

SVC(C=0.1, kernel='linear')

## Test SVM Model

In [82]:
y_pred = svm_model.predict(X_test_scaled)

### Obtain Accuracy of SVM Model

In [83]:
accuracy = accuracy_score(y_test, y_pred)
#print(f"Accuracy: {accuracy * 100:.2f}%")

### Obtain Training and Test Accuracy of SVM Model

In [84]:
train_accuracy = svm_model.score(X_train_scaled, y_train)
test_accuracy = svm_model.score(X_test_scaled, y_test)
#print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
#print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

### Obtain F1 Score of SVM Model

In [85]:
f1 = f1_score(y_test, y_pred)
#print(f"F1 Score: {f1:.2f}")

### Cross Validate SVM Model

In [86]:
scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
#print(f"Cross-Validation Accuracy: {scores.mean():.2f} (+/- {scores.std():.2f})")

## Dump Model to Pickle File

In [87]:
with open('svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)